# Optimal Confidence Threshold Detection with Perch V2

Perch gives every detection a confidence score — but *how high* does a score have to
be before you trust it? That cutoff is species-specific: a 0.3 for one bird can be as
reliable as a 0.7 for another.

This notebook estimates, **per species**, the confidence threshold at which a detection
has a chosen probability (default **95 %**) of being correct. It is self-contained: it
downloads Perch V2 from Kaggle and calls the model directly — no `perchlab` code needed.

## Workflow

1. **Download** Perch V2 from Kaggle with `kagglehub` (cached after the first run).
2. **Load** the TensorFlow SavedModel and its bundled list of species labels.
3. Point it at your **human-validated** clips — the ones a person listened to and
   sorted into `correct/` and `incorrect/` folders.
4. For each clip: load & resample to mono 32 kHz → cut into 5-second windows →
   peak-normalize → **run the model** → keep the confidence given to the *target
   species* by its strongest window.
5. **Fit** a logistic regression of correctness on the logit-transformed confidence,
   per species, and **solve** for the confidence at your target probability.
6. **Save** a thresholds table, a precision-by-confidence-band table, a plot, and a
   `report.md`.

## Method

Wood et al. 2023; validated against Bota et al. 2023.

1. Back-transform each confidence to the logit scale `L = ln(c / (1 − c))`.
2. Fit `P(correct) = sigmoid(b0 + b1·L)` by maximum likelihood.
3. Invert it for the confidence `c*` where `P(correct) = target`:
   `L* = (logit(target) − b0) / b1`, then `c* = sigmoid(L*)`.

### Considerations

- **You need validated clips first.** This notebook does not find birds; it calibrates
  the scores of detections a human has already judged. Produce them with the species
  identification notebook (its segment-extraction step), listen to them, and sort them
  into `correct/` and `incorrect/`.
- **Perch V2 is not amplitude-invariant.** Before inference, each 5-second window must
  have its DC offset removed and its peak amplitude scaled to **0.25** — Perch's
  official preprocessing. Skipping it still returns *an* answer, but the confidence
  scores come out systematically too low, which would bias every threshold here. The
  `peak_normalize` function below does exactly this step.
- **Sample size matters.** A logistic fit on a handful of clips is meaningless. Aim for
  a few dozen clips per species, spread across the confidence range and including both
  correct and incorrect examples.

## Before you run

**Python 3.11 or 3.12** — TensorFlow and Perch do not support 3.13+.

### 1 · Install the libraries

This notebook needs the **TensorFlow** backend. The `onnx` extra does **not** include
TensorFlow, and the import cell below will fail with `ModuleNotFoundError` if you use it.

Inside the PerchLab repo:

```bash
uv venv --python 3.11
source .venv/bin/activate
uv pip install -e ".[tf]"     # TensorFlow (CPU)  —  or ".[gpu]" for CUDA
```

This notebook is self-contained, so you can also run it without the repo:

```bash
pip install "tensorflow>=2.20,<3.0" "librosa>=0.11,<0.12" "kagglehub>=1.0,<2.0" \
            "numpy>=2.0" "pandas>=2.1,<3.0" "scikit-learn>=1.4" "matplotlib>=3.6" notebook
```

### 2 · System audio codecs

Linux/WSL — needed to decode `.mp3` / `.m4a` / `.ogg`:

```bash
sudo apt-get install -y libsndfile1 ffmpeg
```

### 3 · A Kaggle account

Perch V2 is downloaded from Kaggle on the first run — **~780 MB**, cached under
`~/.cache/kagglehub` afterwards, so it only happens once.

1. Accept the model's terms once on the
   [model page](https://www.kaggle.com/models/google/bird-vocalization-classifier).
2. Create a token: Kaggle → *Settings* → *API* → *Create New Token*.
3. Export it **before** launching Jupyter:

```bash
export KAGGLE_API_TOKEN=KGAT_xxxxxxxxxxxxxxxxxxxxxxxx
```

A classic `~/.kaggle/kaggle.json`, or `KAGGLE_USERNAME` + `KAGGLE_KEY`, works too.

### 4 · Point §4 at your own folders

The defaults are **relative** paths resolved from wherever you launched Jupyter. They
almost certainly do not match your machine — the preview cell prints where they actually
landed so you can correct them.

## 0 · Libraries

In [ ]:
import os
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")   # quiet TensorFlow's start-up logs

import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import librosa
import tensorflow as tf
from sklearn.linear_model import LogisticRegression

print("TensorFlow", tf.__version__, "— libraries ready.")

## 1 · Download Perch V2 from Kaggle

`kagglehub` fetches the model the **first** time and caches it under
`~/.cache/kagglehub` for every run afterwards. The first download needs a Kaggle
API token in your environment (create one at Kaggle → *Settings* → *API*, then
`export KAGGLE_API_TOKEN=...` before launching Jupyter).

We use the **`perch_v2_cpu`** build, which runs on any machine. (The plain
`perch_v2` build is compiled for CUDA and will not run on a CPU.)

In [ ]:
import kagglehub

MODEL_SLUG = "google/bird-vocalization-classifier/tensorFlow2/perch_v2_cpu"

model_path = Path(kagglehub.model_download(MODEL_SLUG))
print("Model files at:", model_path)

## 2 · Load the model and the species labels

A Perch SavedModel is called through its `serving_default` signature: you hand it a
batch of 5-second windows shaped `[N, 160000]` (float32) and it returns, among
other things, a `label` array of shape `[N, 14795]` — one raw score (*logit*) per
species, per window.

The species names live in `assets/labels.csv` inside the downloaded model. Its
first line is a namespace tag, so we skip it; the remaining 14,795 names line up
one-to-one with the `label` scores. `CLASS_INDEX` inverts that list so we can look
up a species' column by its scientific name.

In [ ]:
model = tf.saved_model.load(str(model_path))
infer = model.signatures["serving_default"]   # inputs=[N, 160000] float32  ->  {"label", "embedding", ...}

SAMPLE_RATE    = 32_000                        # Perch's required input rate (Hz)
WINDOW_S       = 5.0                           # Perch's fixed window; do NOT change
WINDOW_SAMPLES = int(WINDOW_S * SAMPLE_RATE)   # 160000 samples per window

label_lines = (model_path / "assets" / "labels.csv").read_text().splitlines()
CLASS_NAMES = [ln.strip() for ln in label_lines[1:] if ln.strip()]   # skip the header line
CLASS_INDEX = {name: i for i, name in enumerate(CLASS_NAMES)}        # scientific name -> column
print(f"Loaded {len(CLASS_NAMES)} species classes. First few: {CLASS_NAMES[:3]}")

## 3 · Preprocessing

Three small functions:

- **`load_audio`** — read any common audio file as mono and resample to 32 kHz.
- **`frame_windows`** — slice a clip into 5-second windows, stepping forward by
  `hop_s` seconds each time. Clips shorter than 5 s are padded; a leftover tail
  shorter than one window is dropped.
- **`peak_normalize`** — for each window, subtract the mean (remove DC offset) and
  scale so the loudest sample sits at 0.25.

In [ ]:
def load_audio(path):
    """Load an audio file as mono float32 at Perch's 32 kHz rate."""
    audio, _ = librosa.load(str(path), sr=SAMPLE_RATE, mono=True)
    return audio.astype(np.float32)


def frame_windows(audio, hop_s):
    """Cut audio into Perch's 5 s windows, stepping by hop_s seconds.

    Returns an array of shape [n_windows, 160000].
    """
    hop_samples = int(hop_s * SAMPLE_RATE)
    if len(audio) < WINDOW_SAMPLES:
        audio = librosa.util.pad_center(audio, size=WINDOW_SAMPLES, axis=-1)
    frames = librosa.util.frame(
        audio, frame_length=WINDOW_SAMPLES, hop_length=hop_samples, axis=-1
    ).swapaxes(-1, -2)
    return frames


def peak_normalize(frames, target_peak=0.25):
    """Perch's required per-window preprocessing: remove DC, scale peak to 0.25.

    Perch V2 is NOT amplitude-invariant, so skipping this yields wrong (too-low)
    confidence scores. Silent windows (peak 0) are left at zero.
    """
    frames = frames.astype(np.float32).copy()
    frames -= frames.mean(axis=-1, keepdims=True)
    peak = np.max(np.abs(frames), axis=-1, keepdims=True)
    frames = np.divide(frames, peak, out=np.zeros_like(frames), where=peak > 0.0)
    return frames * target_peak

## 4 · Choose your validated dataset & output folder

- **`VALID_DIR`** — a folder of clips you have already listened to and sorted.
- **`OUTPUT_DIR`** — where the tables, plot, and report are written (created
  automatically).

Two layouts are accepted.

**One species** — `correct/` and `incorrect/` directly inside `VALID_DIR` (then set
`TARGET_SPECIES` in §5 to that bird's scientific name):

```
VALID_DIR/
  correct/     REC01_20250615_054500_Acrocephalus scirpaceus_0.62_15s.wav
  incorrect/   ...
```

**Several species** — one folder per scientific name, each with its own `correct/`
and `incorrect/` (leave `TARGET_SPECIES = None`):

```
VALID_DIR/
  Acrocephalus scirpaceus/{correct,incorrect}/...
  Cettia cetti/{correct,incorrect}/...
```

Folder names are case-insensitive (`correct` / `present` / `true` / `tp` / `yes` =
correct; `incorrect` / `absent` / `false` / `fp` / `no` = incorrect). Species folders
must use the **scientific name** exactly as Perch spells it, since that is how the
class column is looked up.

In [ ]:
# Relative paths are resolved from the folder you launched Jupyter in; the preview
# cell below prints where they actually landed. Absolute paths work too, e.g.
# Path("/home/you/validated") or Path(r"C:/Users/you/validated").
VALID_DIR  = Path("data/validated")                   # <-- change me: your sorted clips
OUTPUT_DIR = Path("data/outputs/optimal_threshold")   # <-- change me (optional): where results go

AUDIO_EXTS = {".wav", ".flac", ".ogg", ".mp3", ".m4a", ".aiff", ".aif"}

def discover_audio(root):
    """Return sorted audio files anywhere under `root`."""
    return sorted(p for p in Path(root).rglob("*") if p.suffix.lower() in AUDIO_EXTS)

Confirm the clips are found, and that the folder layout looks the way you expect, before running the model.

In [ ]:
if not VALID_DIR.exists():
    print(f"⚠ Validated dataset folder not found: {VALID_DIR.resolve()}")
    print("  Point VALID_DIR at a real folder of sorted clips and re-run this cell.")
else:
    found = discover_audio(VALID_DIR)
    print(f"Found {len(found)} audio file(s) under {VALID_DIR.resolve()}\n")
    for sub in sorted(p for p in VALID_DIR.iterdir() if p.is_dir()):
        n = len(discover_audio(sub))
        kids = sorted(d.name for d in sub.iterdir() if d.is_dir())
        print(f"  {sub.name}/  — {n} clip(s)" + (f"  subfolders: {kids}" if kids else ""))

## 5 · Options

- **`TARGET_SPECIES`** — the scientific name for the one-species layout. Leave `None`
  for the multi-species layout (each subfolder names its own species).
- **`TARGET_PROBABILITY`** — the probability of being correct that the threshold is
  solved for. 0.95 is the convention from the literature.
- **`BIN_EDGES`** — lower edges of the confidence bands in the precision table.
- **`ACTIVATION`** — `"softmax"` (the default, and what Perch was trained with) turns
  the logits into probabilities across all species. `"sigmoid"` scores each class
  independently; Perch's winning logits are large, so it saturates near 1.0 and is not
  recommended.
- **`MIN_SAMPLES`** — fewest validated clips before a species is fitted at all.
- **`HOP_S`** — step between the 5 s windows within a clip. Clips are usually about one
  window long, so this rarely matters; 5.0 gives back-to-back windows.
- **`BATCH_WINDOWS`** — how many windows go to the model at once. Lower it if you run
  out of memory.

In [ ]:
TARGET_SPECIES     = None       # scientific name for the one-species layout; None = one per subfolder
TARGET_PROBABILITY = 0.95       # probability of "correct" the threshold is solved for
BIN_EDGES          = [0.1, 0.3, 0.5]   # lower edges of the precision-table confidence bands
ACTIVATION         = "softmax"  # "softmax" (default, matches Perch's training) or "sigmoid" (per-class)
MIN_SAMPLES        = 8          # fewest validated clips needed before a species is fitted

HOP_S         = 5.0   # step between 5 s windows within a clip (s)
BATCH_WINDOWS = 16    # windows sent to the model per batch; lower this if memory is tight

## 6 · Load the validated clips

Walk `VALID_DIR`, read each clip's verdict from the folder it sits in, and preview how
many correct/incorrect clips were found per species.

In [ ]:
CORRECT_NAMES   = {"correct", "present", "true", "tp", "1", "yes", "positive"}
INCORRECT_NAMES = {"incorrect", "absent", "false", "fp", "0", "no", "negative"}


def _verdict(name):
    """Map a folder name to True (correct) / False (incorrect) / None (neither)."""
    key = name.strip().lower()
    if key in CORRECT_NAMES:
        return True
    if key in INCORRECT_NAMES:
        return False
    return None


def _has_verdict_subdirs(directory):
    """True if `directory` has at least one correct/ or incorrect/ subfolder."""
    return any(_verdict(d.name) is not None for d in Path(directory).iterdir() if d.is_dir())


def _collect_species(species_dir, species):
    """Every validated clip under one species folder, tagged correct/incorrect."""
    found = []
    for sub in sorted(Path(species_dir).iterdir()):
        verdict = _verdict(sub.name) if sub.is_dir() else None
        if verdict is None:
            continue
        for path in discover_audio(sub):          # discover_audio defined in §4
            found.append({"path": path, "species": species, "correct": verdict})
    return found


def load_validated_dataset(root, species=None):
    """Discover validated clips under `root` (single- or multi-species layout)."""
    root = Path(root)
    if not root.is_dir():
        raise SystemExit(f"Validated dataset folder not found: {root}")
    if _has_verdict_subdirs(root):                # one-species layout
        if not species:
            raise SystemExit("Found correct/incorrect at the top level; set TARGET_SPECIES.")
        files = _collect_species(root, species)
    else:                                         # multi-species layout
        files = []
        for species_dir in sorted(p for p in root.iterdir() if p.is_dir()):
            if not _has_verdict_subdirs(species_dir):
                print(f"  skipping '{species_dir.name}': no correct/incorrect subfolders")
                continue
            files += _collect_species(species_dir, species or species_dir.name)
    if not files:
        raise SystemExit(f"No validated clips found under {root} (need correct/ and incorrect/).")
    return files


validated = load_validated_dataset(VALID_DIR, TARGET_SPECIES)
val_df = pd.DataFrame([{"species": f["species"], "correct": f["correct"]} for f in validated])
print(f"Loaded {len(validated)} validated clips across {val_df['species'].nunique()} species:")
print(val_df.groupby("species")["correct"].agg(n="size", correct="sum"))

## 7 · Score each clip with Perch

For each clip, run Perch (reusing §3) and keep the confidence it gives the **target
species** — taking the strongest window, the one that drove the detection.
`ACTIVATION` picks softmax (default) or per-class sigmoid.

This is the slow step: every clip goes through the model. Progress prints per clip.

In [ ]:
def window_logits(frames):
    """Raw Perch logits for the windows, scored in batches of BATCH_WINDOWS."""
    out = []
    for start in range(0, len(frames), BATCH_WINDOWS):
        batch = frames[start:start + BATCH_WINDOWS]
        out.append(infer(inputs=tf.constant(batch, dtype=tf.float32))["label"].numpy())
    return np.concatenate(out, axis=0)


def clip_target_confidence(path, species_idx, activation):
    """Perch's confidence for one species on a clip = its strongest window's score."""
    frames = peak_normalize(frame_windows(load_audio(path), HOP_S))
    logits = window_logits(frames)
    if activation == "softmax":
        conf = tf.nn.softmax(logits, axis=-1).numpy()[:, species_idx]
    else:  # per-class sigmoid
        conf = 1.0 / (1.0 + np.exp(-np.clip(logits[:, species_idx], -700.0, 700.0)))
    return float(np.max(conf))


scores_by_species = {}          # species -> {"confidence": [...], "correct": [...]}
for j, f in enumerate(validated, start=1):
    idx = CLASS_INDEX.get(f["species"])
    if idx is None:
        raise SystemExit(f"'{f['species']}' is not a Perch class; use the scientific name.")
    print(f"[{j}/{len(validated)}] {f['species']} · {Path(f['path']).name}")
    bucket = scores_by_species.setdefault(f["species"], {"confidence": [], "correct": []})
    bucket["confidence"].append(clip_target_confidence(f["path"], idx, ACTIVATION))
    bucket["correct"].append(bool(f["correct"]))

print("Scored:", {s: len(v["correct"]) for s, v in scores_by_species.items()})

## 8 · Fit the logistic model per species

The statistics below take plain arrays (confidence + correctness), so they run without
the model — you can re-run this section with different options instantly, without
re-scoring the clips.

`fit_species_threshold` fits `P(correct) = sigmoid(b0 + b1·L)`, solves for the
target-probability threshold, builds the precision bins, and computes the fitted curve
plus its 95 % band. Species with too few clips, all-correct / all-incorrect data, or a
non-positive slope are reported but not fitted — the reason lands in `note`.

In [ ]:
EPS = 1e-6


def logit_score(confidence):
    """Standard logit L = ln(c / (1 - c)) (inverse sigmoid), clipped to stay finite."""
    c = np.clip(np.asarray(confidence, dtype=np.float64), EPS, 1.0 - EPS)
    return np.log(c / (1.0 - c))


def sigmoid(eta):
    """Logistic sigmoid, argument clipped to avoid exp overflow."""
    return 1.0 / (1.0 + np.exp(-np.clip(eta, -700.0, 700.0)))


def precision_bins(confidence, correct, bin_edges):
    """Bin detections by confidence; count verified (correct) per band."""
    conf = np.asarray(confidence, dtype=np.float64)
    ok = np.asarray(correct).astype(bool)
    edges = list(bin_edges) + [1.0 + EPS]
    rows = []
    for i in range(len(edges) - 1):
        low, high = edges[i], edges[i + 1]
        in_bin = (conf >= low) & (conf < high)
        is_last = i == len(edges) - 2
        disp_high = min(high, 1.0)
        det = int(in_bin.sum())
        ver = int((in_bin & ok).sum())
        rows.append({
            "category": f"[{low:.2f}, {disp_high:.2f}{']' if is_last else ')'}",
            "detections": det, "verified": ver,
            "precision": ver / det if det else float("nan"),
        })
    return rows


def _fit_logistic(logit, correct):
    """Unregularised MLE fit; returns (intercept, slope, covariance)."""
    model = LogisticRegression(C=np.inf, solver="lbfgs", max_iter=1000)
    model.fit(logit.reshape(-1, 1), correct.astype(int))
    b0, b1 = float(model.intercept_[0]), float(model.coef_[0][0])
    design = np.column_stack([np.ones_like(logit), logit])
    p = sigmoid(b0 + b1 * logit)
    w = np.clip(p * (1.0 - p), 1e-12, None)
    fisher = design.T @ (design * w[:, None])
    try:
        cov = np.linalg.inv(fisher)
    except np.linalg.LinAlgError:
        cov = np.linalg.pinv(fisher)
    return b0, b1, cov


def fit_species_threshold(species, confidence, correct, target_probability=0.95,
                          bin_edges=(0.1, 0.3, 0.5), min_samples=8):
    """Fit the logistic model and solve for the target-probability threshold."""
    conf = np.clip(np.asarray(confidence, dtype=np.float64), 0.0, 1.0)
    ok = np.asarray(correct).astype(bool)
    n, n_correct = int(conf.size), int(ok.sum())
    n_incorrect = n - n_correct
    result = {
        "species": species, "n": n, "n_correct": n_correct, "n_incorrect": n_incorrect,
        "target_probability": target_probability, "threshold": float("nan"),
        "intercept": float("nan"), "slope": float("nan"), "fitted": False, "note": "",
        "bins": precision_bins(conf, ok, bin_edges),
        "points_conf": conf, "points_correct": ok.astype(float),
        "curve_conf": np.empty(0), "curve_prob": np.empty(0),
        "curve_lo": np.empty(0), "curve_hi": np.empty(0),
    }
    if n < min_samples:
        result["note"] = f"only {n} validated detections (need >= {min_samples})"
        return result
    if n_correct == 0 or n_incorrect == 0:
        only = "correct" if n_incorrect == 0 else "incorrect"
        result["note"] = f"all detections are {only}; cannot fit a regression"
        return result

    logit = logit_score(conf)
    b0, b1, cov = _fit_logistic(logit, ok)
    result["intercept"], result["slope"] = b0, b1
    if b1 <= 0:
        result["note"] = "confidence not positively associated with correctness (slope <= 0)"
        return result

    target_logit = float(np.log(target_probability / (1.0 - target_probability)))
    l_star = (target_logit - b0) / b1
    threshold = float(sigmoid(l_star))              # sigmoid => always in (0, 1)
    result["threshold"], result["fitted"] = threshold, True
    if threshold >= conf.max():
        result["note"] = "threshold exceeds the largest validated confidence (extrapolated)"
    elif threshold <= conf.min():
        result["note"] = "target met below the smallest validated confidence (extrapolated)"

    # Fitted probability curve + 95% band (delta method) for the plot.
    grid = np.linspace(EPS, 1.0 - EPS, 200)
    glogit = logit_score(grid)
    design = np.column_stack([np.ones_like(glogit), glogit])
    eta = b0 + b1 * glogit
    se = np.sqrt(np.clip(np.einsum("ij,jk,ik->i", design, cov, design), 0.0, None))
    result["curve_conf"], result["curve_prob"] = grid, sigmoid(eta)
    result["curve_lo"], result["curve_hi"] = sigmoid(eta - 1.96 * se), sigmoid(eta + 1.96 * se)
    return result


threshold_results = []
for species in sorted(scores_by_species):
    conf = np.asarray(scores_by_species[species]["confidence"], dtype=np.float64)
    correct = np.asarray(scores_by_species[species]["correct"], dtype=bool)
    r = fit_species_threshold(species, conf, correct, target_probability=TARGET_PROBABILITY,
                              bin_edges=BIN_EDGES, min_samples=MIN_SAMPLES)
    threshold_results.append(r)
    if r["fitted"]:
        print(f"{species}: threshold = {r['threshold']:.3f}  (n={r['n']}, {r['n_correct']} correct)"
              + (f"  — {r['note']}" if r["note"] else ""))
    else:
        print(f"{species}: no threshold — {r['note']}")

## 9 · Tables: estimated thresholds & precision by confidence band

Two tables, saved to `OUTPUT_DIR` and shown inline: the per-species thresholds, and the
precision breakdown per confidence band (the totals row lets you sanity-check the fit
against what you actually verified).

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# --- thresholds table (one row per species) ---------------------------------
thr_rows = [{
    "species": r["species"],
    "threshold": round(r["threshold"], 4) if r["fitted"] else None,
    "target_probability": r["target_probability"],
    "n": r["n"], "n_correct": r["n_correct"], "n_incorrect": r["n_incorrect"],
    "total_precision_pct": round(100 * r["n_correct"] / r["n"], 1) if r["n"] else None,
    "intercept": round(r["intercept"], 4) if r["fitted"] else None,
    "slope": round(r["slope"], 4) if r["fitted"] else None,
    "fitted": r["fitted"], "note": r["note"],
} for r in threshold_results]
thresholds_df = pd.DataFrame(thr_rows)
thresholds_df.to_csv(OUTPUT_DIR / "thresholds.csv", index=False)
(OUTPUT_DIR / "thresholds.json").write_text(json.dumps(thr_rows, indent=2), encoding="utf-8")

# --- precision-by-confidence-band table -------------------------------------
prec_rows = []
for r in threshold_results:
    for b in r["bins"]:
        prec_rows.append({
            "species": r["species"], "confidence_category": b["category"],
            "detections": b["detections"], "verified": b["verified"],
            "precision_pct": round(100 * b["precision"], 1) if b["detections"] else None,
        })
    tot_det = sum(b["detections"] for b in r["bins"])
    tot_ver = sum(b["verified"] for b in r["bins"])
    prec_rows.append({
        "species": r["species"], "confidence_category": "TOTAL",
        "detections": tot_det, "verified": tot_ver,
        "precision_pct": round(100 * tot_ver / tot_det, 1) if tot_det else None,
    })
precision_df = pd.DataFrame(prec_rows)
precision_df.to_csv(OUTPUT_DIR / "precision_table.csv", index=False)

print(f"Wrote thresholds.csv, thresholds.json, precision_table.csv -> {OUTPUT_DIR}")
display(thresholds_df)
display(precision_df)

## 10 · Plot: probability of a correct detection

One panel per species: the validated clips as a 0/1 scatter (jittered so overlapping
points stay visible), the fitted logistic curve with its 95 % band, the
target-probability line, and the estimated threshold where the curve crosses it.

In [ ]:
def plot_probability_curves(results, path):
    """Per-species panel: 0/1 scatter, fitted curve + 95% band, target & threshold lines."""
    n = max(1, len(results))
    ncols = min(2, n)
    nrows = (n + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 4.2 * nrows), squeeze=False)
    rng = np.random.default_rng(0)
    flat = axes.ravel()
    for ax, r in zip(flat, results, strict=False):
        jitter = rng.uniform(-0.03, 0.03, size=r["points_correct"].shape)
        ax.scatter(r["points_conf"], r["points_correct"] + jitter, s=10, color="black",
                   alpha=0.5, zorder=3)
        if r["fitted"] and r["curve_conf"].size:
            ax.plot(r["curve_conf"], r["curve_prob"], color="tab:blue", lw=2, zorder=2)
            ax.fill_between(r["curve_conf"], r["curve_lo"], r["curve_hi"],
                            color="tab:blue", alpha=0.15, zorder=1)
            ax.axhline(r["target_probability"], color="gray", ls=":", lw=1)
            if np.isfinite(r["threshold"]):
                ax.axvline(r["threshold"], color="tab:red", ls="--", lw=1.5, zorder=4)
                ax.annotate(f"threshold = {r['threshold']:.2f}", xy=(r["threshold"], 0.5),
                            xytext=(6, 0), textcoords="offset points", color="tab:red",
                            fontsize=8, rotation=90, va="center")
        ax.set_xlim(0, 1)
        ax.set_ylim(-0.08, 1.08)
        ax.set_xlabel("Confidence score")
        ax.set_ylabel("Prob. correct detection")
        ax.set_title(r["species"] if r["fitted"] else f"{r['species']} (no fit)", fontsize=10)
        ax.grid(True, alpha=0.25)
    for ax in flat[len(results):]:
        ax.axis("off")
    fig.tight_layout()
    fig.savefig(path, dpi=120)
    return fig


fig = plot_probability_curves(threshold_results, OUTPUT_DIR / "probability_curves.png")
plt.show()
print(f"Saved plot -> {OUTPUT_DIR / 'probability_curves.png'}")

## 11 · A `report.md`

Bundles both tables and the figure into one human-readable Markdown file, next to the
CSVs — the thing to hand to a collaborator.

In [ ]:
target = threshold_results[0]["target_probability"] if threshold_results else TARGET_PROBABILITY
lines = [
    "# Optimal Confidence Threshold", "",
    f"Confidence threshold for a **{target:.0%} probability of correct identification**, "
    "estimated per species by logistic regression of human-validated detections on the "
    "logit-transformed confidence score.", "",
    "## Estimated thresholds", "",
    "| Species | Threshold | n | Correct | Incorrect | Overall precision |",
    "| --- | --- | --- | --- | --- | --- |",
]
for r in threshold_results:
    thr = f"**{r['threshold']:.2f}**" if r["fitted"] else "—"
    prec = f"{100 * r['n_correct'] / r['n']:.1f}%" if r["n"] else "—"
    lines.append(f"| {r['species']} | {thr} | {r['n']} | {r['n_correct']} | {r['n_incorrect']} | {prec} |")
    if r["note"]:
        lines.append(f"|   ↳ *{r['note']}* | | | | | |")
lines += ["", "## Precision by confidence category", ""]
for r in threshold_results:
    lines += [f"### {r['species']}", "",
              "| Confidence category | Detections | Verified | Precision |",
              "| --- | --- | --- | --- |"]
    for b in r["bins"]:
        prec = f"{100 * b['precision']:.1f}%" if b["detections"] else "—"
        lines.append(f"| {b['category']} | {b['detections']} | {b['verified']} | {prec} |")
    tot_det = sum(b["detections"] for b in r["bins"])
    tot_ver = sum(b["verified"] for b in r["bins"])
    tot_prec = f"{100 * tot_ver / tot_det:.1f}%" if tot_det else "—"
    lines += [f"| **TOTAL** | {tot_det} | {tot_ver} | {tot_prec} |", ""]
lines += ["## Figure", "", "![Probability of correct detection](probability_curves.png)", ""]
(OUTPUT_DIR / "report.md").write_text("\n".join(lines), encoding="utf-8")
print(f"Wrote report.md -> {OUTPUT_DIR / 'report.md'}")

## 12 · Using the thresholds

`thresholds.csv` now holds one confidence cutoff per species. Back in the species
identification notebook, a detection of *Acrocephalus scirpaceus* below that species'
threshold is one you should not report without listening to it — and one above it is
correct about 95 % of the time (or whatever `TARGET_PROBABILITY` you set).

A threshold is only as good as the clips it was fitted on: it applies to recordings
like the ones you validated (same site, season, recorder, noise floor). Re-fit when the
dataset changes.